# 04: Member C Wish dataset experiment

Dataset: [RealVsFake_162k_by_Wish](https://www.kaggle.com/datasets/wish096/realvsfake-81k-by-wish).

The first cell connects this exact dataset through KaggleHub. On Kaggle it attaches the dataset as notebook input; manually attaching it first is also fine. Enable Internet for the initial dependency/weight downloads and select a GPU for training. A dataset connection is not proof that training has passed.

Use the same dataset version and split manifest as Member A. No independent random split is generated. Run section by section; final evaluation is deliberately gated.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json

DATASET_HANDLE = "wish096/realvsfake-81k-by-wish"
DATASET_VERSION = None  # set to Member A's positive integer Kaggle version before auditing/training
MEMBER_A_CSV = None     # set to the uploaded shared manifest path, not the dataset URL
MEMBER_A_SPLIT_CSVS = {} # alternatively: {"train": ".../train_labels.csv", "val": ".../val_labels.csv", "test": ".../test_labels.csv", "cross_gen": ".../cross_gen_labels.csv"}
MEMBER_A_PATH_PREFIX = "" # exact prefix to remove if A exported absolute Kaggle paths
NEAR_DUPLICATE_REVIEW = None  # optional path to the documented human-review JSON
REPO_OVERRIDE = None   # optional path to an uploaded/checked-out repository
OUTPUT_OVERRIDE = None # optional persistent writable output directory

def run(*args):
    subprocess.run([sys.executable, *map(str, args)], check=True)

def wish_handle(version):
    if version is None:
        return DATASET_HANDLE
    if isinstance(version, bool) or not isinstance(version, int) or version < 1:
        raise ValueError("DATASET_VERSION must be a positive integer from Member A.")
    return f"{DATASET_HANDLE}/versions/{version}"

def find_wish_root(directory):
    directory = Path(directory)
    roots = []
    # Inspect directories only, including nested RealVsFake/RealVsFake layouts.
    for current, subdirs, _ in os.walk(directory):
        if "Real" in subdirs and "Fake" in subdirs:
            roots.append(Path(current).resolve())
            subdirs[:] = []
        else:
            subdirs[:] = [name for name in subdirs if not name.startswith(".")]
    if len(roots) != 1:
        raise RuntimeError(f"Expected one Real/Fake image root in {directory}; found {roots}")
    return roots[0]

try:
    import kagglehub
except ModuleNotFoundError:
    run("-m", "pip", "install", "kagglehub")
    import kagglehub

REQUESTED_HANDLE = wish_handle(DATASET_VERSION)
DATASET_DIR = Path(kagglehub.dataset_download(REQUESTED_HANDLE))
DATA_ROOT = find_wish_root(DATASET_DIR)
print("Dataset:", REQUESTED_HANDLE)
print("Resolved images:", DATA_ROOT)
if DATASET_VERSION is None:
    print("Connected to the latest available release for inspection only. Set Member A's exact version and rerun this cell before training.")


## Repository code

The notebook imports the project modules. Upload/extract the latest repository under `/kaggle/working/deepfake-detection`, or clone your branch after its latest changes have been pushed. The notebook does not publish or push code for you. A locally committed but unpushed implementation is not available to Kaggle merely from its GitHub URL.


In [ ]:
candidates = ([Path(REPO_OVERRIDE)] if REPO_OVERRIDE else [
    Path.cwd(), Path.cwd().parent,
    Path("/kaggle/working/deepfake-detection"), Path("/content/deepfake-detection"),
])
REPO = next((p.resolve() for p in candidates
             if (p / "models/vit/manifest.py").is_file()
             and (p / "requirements-member-c.txt").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Dataset loaded, but project code is missing. Upload/extract the current repository and set REPO_OVERRIDE.")
os.chdir(REPO)
OUTPUT_ROOT = (Path(OUTPUT_OVERRIDE) if OUTPUT_OVERRIDE else
               (Path("/kaggle/working/member_c") if Path("/kaggle/working").is_dir()
                else REPO / "results/notebook"))
OUTPUT_ROOT = OUTPUT_ROOT.resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Code:", REPO)
print("Outputs:", OUTPUT_ROOT)


In [ ]:
run("-m", "pip", "install", "-r", "requirements-member-c.txt")
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
run("-m", "pytest", "-q")


## Shared split handoff

Use either `MEMBER_A_CSV` for one combined manifest OR `MEMBER_A_SPLIT_CSVS` for A's separate train/val/test/cross_gen files. Set these and the integer `DATASET_VERSION` in the first cell, then rerun it. Separate files require `filepath,label`; source is inferred from the documented RFF/RCA/FSG/FSD/AI prefixes only when the source column is absent. Existing source labels are validated, never replaced.

Paths must be relative to the detected Real/Fake parent. For old absolute CSV paths, set `MEMBER_A_PATH_PREFIX` to the exact prefix ending before `Real/` or `Fake/`. Row assignments are preserved. If A's three-way splits put Stable Diffusion in training or do not provide separate real controls for cross_gen, the notebook stops for a shared correction by A; it does not silently move images.


In [ ]:
if DATASET_VERSION is None or REQUESTED_HANDLE != wish_handle(DATASET_VERSION):
    raise ValueError("Set Member A's exact DATASET_VERSION and rerun the dataset loading cell first.")
if MEMBER_A_CSV and MEMBER_A_SPLIT_CSVS:
    raise ValueError("Choose a combined manifest OR separate CSV files, not both.")
HANDOFF_CSV = MEMBER_A_CSV
if MEMBER_A_SPLIT_CSVS:
    from models.vit.manifest import combine_split_csvs
    HANDOFF_CSV = combine_split_csvs(MEMBER_A_SPLIT_CSVS, OUTPUT_ROOT / "handoff/member_a.csv",
                                      strip_prefix=MEMBER_A_PATH_PREFIX)
if not HANDOFF_CSV or not Path(HANDOFF_CSV).is_file():
    raise FileNotFoundError("Attach Member A's shared CSV and set MEMBER_A_CSV in the first cell.")
HANDOFF_CSV = Path(HANDOFF_CSV).resolve()
from models.vit.manifest import load_audited_manifest, sha256_file
AUDIT = OUTPUT_ROOT / f"data_audit/wish_v{DATASET_VERSION}"
if not AUDIT.exists():
    run("-m", "models.vit.prepare_data", "--manifest", HANDOFF_CSV,
        "--data-root", DATA_ROOT, "--dataset-version", DATASET_VERSION, "--output", AUDIT)
audit = json.loads((AUDIT / "audit.json").read_text())
if (audit["input_manifest_sha256"] != sha256_file(HANDOFF_CSV)
        or audit["dataset_version"] != str(DATASET_VERSION)):
    raise ValueError("Existing audit belongs to another handoff. Choose a new OUTPUT_OVERRIDE; do not reuse stale results.")
review = str(Path(NEAR_DUPLICATE_REVIEW).resolve()) if NEAR_DUPLICATE_REVIEW else None
_, audit = load_audited_manifest(AUDIT / "manifest.csv", DATA_ROOT, review)
print(json.dumps(audit, indent=2))


## Automatic runtime configuration

This cell writes separate runtime YAML files into the output directory. It does not edit tracked team configs or split assignments. All model configs point to the same audited manifest. Save the complete output folder before the GPU session ends. Resume requires the original run/configs; a new experiment needs a new output directory or run name.


In [ ]:
import yaml
CONFIG_DIR = OUTPUT_ROOT / "configs"
CONFIG_DIR.mkdir(exist_ok=True)
sources = {
    "custom_cnn": "configs/wish/custom_cnn.yaml",
    "resnet50": "configs/wish/resnet50.yaml",
    "efficientnetv2": "configs/wish/efficientnetv2.yaml",
    "vit_b16": "configs/vit.yaml",
}
CONFIGS = {}
harness = {"manifest": str(AUDIT / "manifest.csv"), "data_root": str(DATA_ROOT),
           "near_duplicate_review": review, "models": {}}

def write_runtime_yaml(path, value):
    if path.exists() and yaml.safe_load(path.read_text()) != value:
        raise ValueError(f"Existing run configuration differs: {path}. Use a new OUTPUT_OVERRIDE.")
    path.write_text(yaml.safe_dump(value, sort_keys=False))

for name, source in sources.items():
    cfg = yaml.safe_load(Path(source).read_text())
    cfg["data"].update(root=str(DATA_ROOT), manifest=str(AUDIT / "manifest.csv"),
                       near_duplicate_review=review)
    cfg["output"]["results_dir"] = str(OUTPUT_ROOT / "runs" / name)
    CONFIGS[name] = CONFIG_DIR / f"{name}.yaml"
    write_runtime_yaml(CONFIGS[name], cfg)
    checkpoint = Path(cfg["output"]["results_dir"]) / cfg["run_name"] / "checkpoints/best_model.pt"
    harness["models"][name] = {"config": str(CONFIGS[name]), "checkpoint": str(checkpoint)}
HARNESS_CONFIG = CONFIG_DIR / "crossgen_harness.yaml"
write_runtime_yaml(HARNESS_CONFIG, harness)
FROZEN = OUTPUT_ROOT / "comparison/frozen.json"
FINAL = OUTPUT_ROOT / "comparison/final"
print("Configured training and evaluation with:", DATA_ROOT)
print("ViT config:", CONFIGS["vit_b16"])


In [ ]:
assert torch.cuda.is_available(), "Select a GPU runtime for real ViT training."
run("-m", "models.vit.train", "-c", CONFIGS["vit_b16"], "--smoke")


In [ ]:
# A/B run their configurations separately using the SAME audited handoff.
run("-m", "models.vit.train", "-c", CONFIGS["vit_b16"])
# After interruption, resume with the same config and --resume <run>/checkpoints/last.pt.


## Final comparison, only after all four runs

Place A/B's matching run folders at the checkpoint locations printed in the generated harness, or supply a reviewed harness with their actual checkpoint paths. Choose all settings before opening test results. The default notebook trains ViT only, not A/B's three models. Final evaluation stops if any required model is missing.


In [ ]:
READY_FOR_FINAL_EVALUATION = False  # change only after all four final runs are ready
assert READY_FOR_FINAL_EVALUATION, "Confirm all checkpoints/settings before opening tests."
run("-m", "models.vit.evaluate_crossgen", "freeze", "-c", HARNESS_CONFIG, "--output", FROZEN)
run("-m", "models.vit.evaluate_crossgen", "run", "--frozen", FROZEN, "--output", FINAL, "--device", "cuda")


In [ ]:
import pandas as pd
assert json.loads((FINAL / "status.json").read_text())["status"] == "complete"
display(pd.read_csv(FINAL / "model_comparison.csv"))
display(pd.read_csv(FINAL / "figures/failure_cases.csv").head(20))
run("-m", "models.vit.evaluate_crossgen", "plots", "--predictions", FINAL / "predictions.csv",
    "--output", OUTPUT_ROOT / "comparison/regenerated")


## Report and demo

Follow reports/final_report_draft.md and docs/member_c_delivery.md. Explain observed failures, source/pretraining confounds and limitations. Use python -m models.vit.predict for a cropped-face demo. Do not tune on these final results or claim universal detection.

Dataset loading uses the [official KaggleHub dataset API](https://github.com/Kaggle/kagglehub#download-dataset). Successful local mock tests do not establish live Kaggle connectivity or GPU training success.
